In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import scipy.stats as stats
from scipy.stats import spearmanr, pearsonr, binomtest

sns.set(style='whitegrid')

repo_root = Path('/playpen-ssd/wokwen/projects/autoeval_chatbot')
data_path = repo_root / 'ratings' / 'qwen' / 'cross_eval_ratings.csv'
df = pd.read_csv(data_path, on_bad_lines='skip')
print('Loaded', df.shape, 'rows')

# Detect rating columns for models (non-blinded data should have model-specific rating columns)
rating_cols = [c for c in df.columns if c.endswith('_rating')]
if len(rating_cols) == 0:
    print('No explicit rating columns found (no *_rating). We will try to infer by common names.' )
    # common fallback names (modify if your dataset uses different names)
    for name in ['chatbot_rating','prompt_no_data_rating','data_no_prompt_rating','gpt_rating','llama_rating','qwen_rating']:
        if name in df.columns:
            rating_cols.append(name)
print('Rating columns detected:', rating_cols)

# Detect candidate criteria columns (heuristic)
candidate_patterns = ['_score$', '_criterion$', 'helpful', 'helpfulness', 'accuracy', 'relevance', 'coherence', 'factual', 'correctness', 'fluency']
criteria_cols = []
for c in df.columns:
    lc = c.lower()
    if any(re.search(p, lc) for p in candidate_patterns):
        criteria_cols.append(c)
# remove rating columns if they match the pattern
criteria_cols = [c for c in criteria_cols if c not in rating_cols and not c.lower().endswith('_rating')]
print('Detected criteria columns:', criteria_cols)
if len(criteria_cols) == 0:
    print('No explicit per-criterion columns found. You can still run keyword extraction on reason columns later.')

# Compute per-criterion means and correlations with each model's ratings
crit_means = {}
crit_corrs = {}
if len(criteria_cols) > 0 and len(rating_cols) > 0:
    for crit in criteria_cols:
        crit_series = pd.to_numeric(df[crit], errors='coerce').dropna() if crit in df.columns else pd.Series(dtype=float)
        crit_means[crit] = crit_series.mean() if not crit_series.empty else np.nan
        # correlations with each rating column (only where both present)
        corr_row = {}
        for rcol in rating_cols:
            rseries = pd.to_numeric(df[rcol], errors='coerce')
            common_idx = crit_series.index.intersection(rseries.dropna().index)
            if len(common_idx) > 0:
                cval = spearmanr(crit_series.loc[common_idx], rseries.loc[common_idx]).correlation
            else:
                cval = np.nan
            corr_row[rcol] = cval
        crit_corrs[crit] = corr_row
    crit_means = pd.Series(crit_means).sort_values(ascending=False)
    crit_corrs_df = pd.DataFrame(crit_corrs).T[rating_cols]
    print('Criterion means (global):')
    display(crit_means)




















































443.10.0pythonpython3pythonPython 3print(words.most_common(40))print('Top reason tokens:')    words.update(toks), t.lower()) if len(w) > 3]    toks = [w for w in _re.findall(rfor t in texts:words = Counter()    texts.extend(df[c].dropna().astype(str).tolist())for c in reason_cols:texts = []print('Reason columns found:', reason_cols)reason_cols = [c for c in df.columns if 'reason' in c.lower() or c.lower().endswith('_reason')]# search for reason-like columns in the dataframeimport re as _refrom collections import CounterpythoncodeIf you don't have structured criteria, extract frequent keywords from reason columns to approximate critique categories.### Keyword extraction from free-text reasons (optional)markdownmarkdown    print('No criteria to visualize.')else:        plt.show()        plt.tight_layout()        plt.xlabel('Rating column')        plt.ylabel('Criterion')        plt.title('Spearman correlation: criterion vs model ratings')        sns.heatmap(crit_corrs_df, annot=True, fmt='.2f', cmap='vlag', center=0)        plt.figure(figsize=(max(6, len(rating_cols)*1.2), max(4, len(criteria_cols)*0.4)))    if 'crit_corrs_df' in globals():    plt.show()    plt.tight_layout()    plt.ylabel('Mean')    plt.title('Global criterion means')    crit_means.plot(kind='bar', color='C0')    plt.figure(figsize=(8,4))if len(criteria_cols) > 0:# Visualize criterion means and correlationspythoncode    print('Not enough criteria or rating columns to compute criterion-vs-model correlations.')else:    display(crit_corrs_df)Correlation (Spearman) between criterion and each model rating:')